# freehiero — Detector 測試

比較 **KeywordDetector**（L1）與 **Qwen2.5-0.5B**（L2）在偵測「免費食物」貼文的準確率。

執行環境：Colab（T4 GPU）

## 1. 安裝依賴

In [ ]:
!pip install -q transformers accelerate

## 2. KeywordDetector（直接複製 detector.py 的 L1 邏輯）

In [ ]:
import re

# 與 detector.py 保持一致（自取移除，食物詞擴充）
_FREE_WORDS = r'免費|free|請拿|拿走|多餘|多的|送人|不要了|剩食|剩菜|拿去|有需要|帶走|送出|分享'
_FOOD_WORDS = r'食物|食品|飯|麵|便當|零食|餅乾|水果|蔬菜|菜|湯|肉|蛋|麵包|吐司|料理|點心|糕|餅|粽|飲料|奶茶|咖啡|茶|寶特瓶|三明治|沙拉|漢堡|披薩|壽司|飯糰|泡麵|湯圓'
_PATTERN_FREE_FOOD = re.compile(rf'(?=.*({_FREE_WORDS}))(?=.*({_FOOD_WORDS}))', re.IGNORECASE)
_PATTERN_NOT_FOOD  = re.compile(r'免費.*?(?:課程|諮詢|活動|講座|workshop|票|名額|參加|索取)', re.IGNORECASE)

def keyword_detect(text: str) -> bool:
    if not text:
        return False
    if _PATTERN_NOT_FOOD.search(text):
        return False
    return bool(_PATTERN_FREE_FOOD.search(text))

print('KeywordDetector loaded')

## 3. 測試資料集

**標記說明**：`1` = 確實在送免費食物，`0` = 非食物或不是免費送

> 上線前請補充真實社團貼文（正例 + 負例各 ≥ 15 篇）

In [ ]:
SAMPLES = [
    # ── 正例（label=1）真實社團貼文 ──────────────────────────────────────────────
    # #18 研討會飲料剩餘
    (1, '不好意思又打擾了～\n研討會小強不夠多所以飲料還有剩，歡迎大家來喝～\n地點：航太系館1樓\n種類：冬瓜停檸、無糖四季春青茶、黑糖紅茶'),
    # #20 系統系館研討會多餘便當
    (1, '在系統系館中庭後面的長桌有研討會多出來的便當~~\n要取要快 食安自負'),
    # #36 活動多的披薩（NT$67 = 免費梗，現代年輕人用法）
    (1, '活動多的披薩，食安靖自行負真\n地點：系統系館\n時間：21:00前'),
    # #40 研討會便當
    (1, '研討會多的便當，超級好吃，歡迎前來自取\n地點：航太系館1樓'),
    # #43 典型正例：食物照片 + 免費標題 + 地點 + 食安自負
    (1, '研討會多的便當，超級好吃，歡迎前來自取\n地點：航太系館1樓\n食安自負'),
    # #45 舞展工作人員有免費供餐（付出勞力但有免費食物）
    (1, '成大舞研年度舞展小黑人招募中！\n福利：當日免費供餐！可提早於最佳觀眾席看到舞者們精彩的演出！\n日期：6/27（舞展當日）\n服裝：全身黑色（工作基本dress code）'),
    # #46 餅乾不合口味故送出
    (1, '上禮拜去農會超市買的餅乾 不合口味\n故送出\n青心烏龍茶燒米餅 有七包'),
    # #47 研討會多的日式便當
    (1, '研討會多的日式便當，目前有10個\n太子文旅三樓自取'),
    # #48 贈送貓咪零食餐包（寵物食品亦算）
    (1, '（已預訂）送全新未折貓咪零食跟餐包\n效期都在2027之後（藍莓小貓頭是2026九月）\n家裡的挑食怪連零食都挑 受不了\n送給有需要的人，東平路烏麵包附近自取～'),
    # #49/#50 Meetup 活動有供餐
    (1, '5/12(二)Meetup活動資訊：\n主題：智慧機器人產業應用趨勢\n時間：5/12（二）18:00 ～ 20:00（17:45開始到場）\n地點：陽明交通大學台南校區 奇美樓218教室\n注意事項：有供餐'),
    # ── 負例（label=0）真實社團貼文 ──────────────────────────────────────────────
    # #15 心理量表問卷招募（免費活動，非食物）
    (0, '大家好！我們是成心理系修習【量表編製與評估】課程的學生，目前正在進行「純威力量表」的編製研究，想了解大學生在面對壓力、批評與情緒波動時的反應方式與心理調適傾向。誠摯邀請大家協助填寫問卷囉'),
    # #16 跨感官知覺研究招募
    (0, '各位社團朋友好，我是成大心理系黃君群老師實驗室研究負責人 張良聖。目前實驗室正在進行一項短期學術研究，邀請大家在電腦前勤動手指，協助我們完成線上調查！研究主題：跨感官體驗與美感知覺（味覺/形狀聯結、書法偏好評估）'),
    # #17 化學書鍵盤販售
    (0, '賣舊書賣鍵盤，有些附小贈品，橫批絕處逢生\n微積分、普物、物化、無機、分析化學教科書，統一價800\n化學系本本 -100，被當過主科 -100\n狼蛛 F75雪杉綠鍵盤 700'),
    # #19 南山公墓田野調查問卷
    (0, '大家午安，我們是來自成大台文系的學生，有一門必修小專題，我們是做關於南山公墓與都市更新的研究，急需田野調查資料，希望同學能夠撥冗填寫。填寫時間大約三分鐘！'),
    # #21 免費贈送小玩具（非食物）
    (0, '長樂路二段全聯或南紡購物中心面交\n【免費贈送】一些不玩的小玩具'),
    # #26 畢業大拍賣（FB 亂填免費，其實是有價販售）
    (0, '即將畢業搬家出清！生活好物便宜到愛，數量有限，售完為止！\n大同100L單門小冰箱、宜得利矮立間坐面和坐椅、喜菲久坐不鏽腰坐墊、木紋實用摺疊桌、Apple Pencil 1、HP H100電競耳機、SAMPO聲寶捕蚊燈\n台南市區or成大自強校區面交'),
    # #27 一番賣出清（FB 亂填免費，其實是有價販售）
    (0, '一番賣出清～這次被貓巨破錢包了😅\n價格都在圖上 有問路都歡迎詢問\n多收優先\n南台面交or賣貨便須匯款就+$20\n行李箱只接受面交'),
    # #28 車禍徵目擊者
    (0, '在5/26約早上9點50分時，在長樂路四段5號（新K對面）發生一場車禍，但我的車上沒有安裝行車記錄器，想詢問大家是否有人經過可以提供畫面；我會再請你喝星巴克的'),
    # #29/#30 生育意願問卷
    (0, '大家好，我們是114-2社會心理學的修課學生，目前正在進行課堂報告關於「對於生育意願的態度」的研究調查。填答條件：具中華民國國籍與華文閱讀能力者皆可填答。本問卷不具名，填答時間約3-5分鐘，沒有標準答案'),
    # #31 搬家出清（有標價，已預訂/吊出狀態）
    (0, '即將畢業搬家出清，生活好物便宜到愛，數量有限，售完為止！\n大同100L單門小冰箱 (TR-100S) (已預訂)、宜得利矮立間坐面和坐椅 (已吊出)、喜菲久坐不鏽腰坐墊 (已吊出)、木紋實用摺疊桌 (已吊出)\nApple Pencil 1、HP H100電競耳機、SAMPO聲寶捕蚊燈、雷達薄型液體電蚊香'),
    # #34 雜物娃娃出清（有標價）
    (0, '（暫售）免打孔的桌面上加寬延伸板：50\n排球少年赤葦京治公仔 全新：600\n（暫售）1.8L帶蒸籠萬用鍋 買來一年沒用過：400\nHAPIINS奶酪貓 全新：150\nHAPIINS牛奶貓 全新：150\n超大兔卡提西亞娃娃：500\n黑糖鮮奶麻糬湯材料組合包（全新未開封）：50'),
    # #35 搬家出清 IKEA（有標價）
    (0, '搬家出清！！！ 價格如圖～～～🙏\nIKEA娃娃兩隻一起$400'),
    # #37 協尋鑰匙（失物招領）
    (0, '地點：成大自強校區儀器設備大樓9樓\n近期有到儀器設備大樓B1無塵室做實驗的同學，你的鑰匙掉在無塵衣的口袋裡，被我們撿到，看到貼文的話，請盡快來9樓核心設施中心櫃台認領。'),
    # #38/#39 成心城涌錯覺展演（免費活動，非食物）
    (0, '成心城涌，河以路尋：第三屆成大心理系錯覺展演\n你看到的，真的是你看到的嗎？今年夏天，成功大學心理系與成功大學歷史系及日本立命館大學合作籌辦\n展覽資訊：5/21(四)－5/23(六) 9:00-21:00 @ 成大歷史文物館\n費用：免費'),
    # #41 南山公墓社遊（免費社遊，非食物）
    (0, '（社遊借版宣傳）\n延續上次社社的主題，目前位於台南機場北側，擁有超過400年歷史的南山公墓，目前正面臨被拆除、強制遷葬的危機。本次湯社的社遊將由南山國家墓葬歷史生態區促進會秘書長擔任導覽員\n費用：免費\n時間：5/24（日）14:00-15:30'),
    # #42 徵二手電風扇（徵收，非免費送）
    (0, '大家好，我想買一台二手電風扇。如果有人想轉讓的話，請留言或私訊我價格和狀況。謝謝！'),
    # #44 搬家出清（有標價）
    (0, '搬家出清！！！ 價格如圖～～～🙏\nIKEA娃娃兩隻$400\n各式碗盤 $15/個，共五個全部走$60'),
]

print(f'資料集：{sum(l==1 for l,_ in SAMPLES)} 正例 / {sum(l==0 for l,_ in SAMPLES)} 負例')

## 4. 評估 KeywordDetector

In [ ]:
def evaluate(name, predict_fn, samples):
    tp = fp = tn = fn = 0
    errors = []
    for label, text in samples:
        pred = predict_fn(text)
        if label == 1 and pred:     tp += 1
        elif label == 0 and not pred: tn += 1
        elif label == 0 and pred:
            fp += 1
            errors.append(('FP', text[:60]))
        else:
            fn += 1
            errors.append(('FN', text[:60]))

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f'\n── {name} ──')
    print(f'  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}')
    print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    if errors:
        print('  錯誤案例:')
        for tag, t in errors:
            print(f'    [{tag}] {t}')
    return dict(name=name, precision=precision, recall=recall, f1=f1)

kw_result = evaluate('KeywordDetector (L1)', keyword_detect, SAMPLES)

## 5. Qwen2.5-0.5B-Instruct（L2 模型，需 GPU）

模擬 `OllamaDetector` 的邏輯，但改用 HuggingFace Transformers 直接跑。
這樣不需要 Ollama server，在 Colab 上也能驗證模型品質。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
print(f'模型載入完成，裝置：{next(model.parameters()).device}')

In [ ]:
PROMPT_TMPL = (
    '你是食物偵測助理。判斷以下社團貼文是否在免費提供食物（讓人拿取）。'
    '只回答 yes 或 no，不要解釋。\n\n貼文：{text}'
)

def qwen_detect(text: str) -> bool:
    messages = [
        {'role': 'system', 'content': '你只回答 yes 或 no。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    # transformers 版本不同，apply_chat_template 可能回傳 BatchEncoding 而非純 tensor
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    ids = ids.to(model.device)
    input_len = ids.shape[1]
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip().lower()
    return answer.startswith('yes')

# Quick smoke test
print(qwen_detect('有多的便當，免費拿走，在工程館'))  # expect True
print(qwen_detect('出售二手書'))                       # expect False

In [ ]:
qwen_result = evaluate('Qwen2.5-0.5B (L2)', qwen_detect, SAMPLES)

## 6. 比較結果

In [ ]:
print(f'\n{'模型':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}')
print('-' * 58)
for r in [kw_result, qwen_result]:
    print(f"{r['name']:<25} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")

## 7. 兩層合併效果（L1 miss → L2）

In [ ]:
_ANY_SIGNAL = re.compile(rf'({_FREE_WORDS})|({_FOOD_WORDS})', re.IGNORECASE)

def two_layer_detect(text: str) -> bool:
    if keyword_detect(text):
        return True
    # only send to model if there's at least some signal
    if _ANY_SIGNAL.search(text):
        return qwen_detect(text)
    return False

two_result = evaluate('TwoLayer (L1 + L2)', two_layer_detect, SAMPLES)

print(f'\n{'模型':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}')
print('-' * 58)
for r in [kw_result, qwen_result, two_result]:
    print(f"{r['name']:<25} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")

## 8. 新增自訂測試樣本

把遇到的真實貼文貼進來，看哪層漏掉、原因為何。

In [ ]:
CUSTOM = [
    # 貼上真實貼文測試，格式：(true_label, '貼文內容')
    # (1, '...'),
    # (0, '...'),
]

if CUSTOM:
    print('=== KeywordDetector ===')
    evaluate('KeywordDetector', keyword_detect, CUSTOM)
    print('\n=== Qwen2.5 ===')
    evaluate('Qwen2.5', qwen_detect, CUSTOM)
    print('\n=== TwoLayer ===')
    evaluate('TwoLayer', two_layer_detect, CUSTOM)
else:
    print('CUSTOM 為空，請填入真實貼文後重新執行')